We will now do Linear and Logistic regression from scratch in Python

We will be starting from the basics. And I will be writing every theory and concept with the code.

# Linear Regression Model

Linear regression model is a model in machine learning where the model prdicts the relationship between dependent variable and independent variable.
Simple linear regression: 

$$
y = \beta_0 + \beta_1 x + \varepsilon
$$

In the regression model we approximate the straight line:

$$
\hat{y} = \beta_0 + \beta_1 x 
$$

here x is the imdependent variable. And $\hat{y}$ is the output or dependent variable. We can predict the dependent variable y for the value of the independent variable x.

But the true or the actual value of y can be different (in most cases, will be different) from the predicted value of y.
That difference is presented as **residual** or **error** term.

**Residual:**

$$
e = y - \hat{y}
$$



### Loss Function

After we make prediction with the linear regression model, we need a way to measure how wrong the predictions of that model is. How poorly it performs. This is where loss function comes in.

The first idea to come in mind is that to square each residual and taking their average, which is called **Mean Squared Error**. There are other lost functions (Mean Absolute Error, Huber etc) but MSE is called **The Standard Loss** function.

**MSE:**
$$
J(w, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

Why we square the residuals?
1. Positive and negative residuals can cancel each other out, squaring them prevents this from happening.
2. Larger errors receive higher penalty.
3. The square function is differenciable, and we can differentiate it much easily than absolute functions

Low the MSE means our model predicts the y variable more accurately. The less the MSE the better the model.

We will now implement MSE in Python.


NumPy lets us perform the same calcualtion on the entire array. So basically Numpy array works line R's vectors (I am sounding like Rasel sir :) ). np.mean() will give us the mean of the entire array

In [1]:
import numpy as np 

def mse(list_actual, list_predicted):
    actual = np.array(list_actual)
    predicted  = np.array(list_predicted)

    residual = actual - predicted

    squared_residual = residual**2

    MSE = np.mean(squared_residual)

    return float(MSE) 

mse([10,10], [8,13])
# Here these points are y values (y_actual, y_predicted). the mse gives us the mean of squared (y_actual - y_predicted).

6.5

In [2]:
mse([5,10,15], [4,12,12])

4.666666666666667

Now we have a concept of what MSE is and what a loss function is. We know that the less the MSE the better the model. So the best model will be which has the minimum MSE. We need to find the parameters ($\beta_0$ & $\beta_1$) for which the MSE is minimized. We did it in our class by differentiating the loss function, taking partial derivative with respect to each parameter and the equating it to 0. But here we will use gradient decent for finding out the optimal parameters.

### Gradient Descent

Yt video on this: [Gradient Descent](https://youtu.be/sDv4f4s2SB8?si=tk597KYavM34KHQw) (StatQuest)

In statistics and ML we optimize things. In linear regression we optimize for slope and intercept, in logistic regression we optimize the graph and in t-SNE we optimize clusters. Gradient descent can optimize for all of these. If we learn how to optimize for only linear regression we can do it for the other areas too.


Now that we have a loss function (MSE) that tells us how wrong our model is, we need a way to actually reduce that loss. We can't just guess values of $\beta_1$ and $\beta_0$  forever — we need a systematic way to move toward better ones. This is where **gradient descent** comes in.

The idea is simple: the loss function $J(\beta_1 , \beta_0)$ is like a surface with hills and valleys. We want to find the lowest point — the values of $/beta_1$  and $\beta_0$ where the loss is smallest. Gradient descent does this by repeatedly taking small steps in the direction that decreases the loss the fastest.

**How do we know which direction decreases the loss?**

That's what the **gradient** tells us. The gradient is just the derivative of the loss function with respect to each parameter — it tells us the slope of the loss surface at our current position. If the slope is positive, the loss increases as the parameter increases, so we should move the parameter in the opposite direction. If the slope is negative, we should increase it.

**Gradients of MSE:**
$$
\frac{\partial J}{\partial \beta_1} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot x_i
$$
$$
\frac{\partial J}{\partial \beta_0} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)
$$

**Update rule:**
$$
\beta_1 := \beta_1 - \alpha \frac{\partial J}{\partial \beta_1}, \qquad \beta_0 := \beta_0 - \alpha \frac{\partial J}{\partial \beta_0}
$$

Here, $\alpha$ (alpha) is the **learning rate** — it controls how big a step we take at each update.

Why do we subtract the gradient instead of adding it?
1. The gradient always points in the direction of **steepest increase** of the loss.
2. Since we want to *minimize* the loss, we move in the opposite direction — hence the minus sign.
3. Repeating this process moves $\beta_1$  and $\beta_0$ a little closer to the values that minimize $J(\beta_1, \beta_0)$ each time.

**Why does the learning rate matter?**
1. If $\alpha$ is too small, the model learns extremely slowly, taking tiny steps toward the minimum.
2. If $\alpha$ is too large, the updates can overshoot the minimum and the loss may oscillate or even diverge instead of settling down.
3. A well-chosen $\alpha$ lets the loss decrease steadily, epoch after epoch, until it levels off near its lowest point.

We repeat this update for many **epochs** (passes over the training data/repeatation). Each epoch, $\beta_1$  and $\beta_0$ shift a little further downhill, and the MSE should get smaller and smaller — until it converges, meaning further updates barely change the loss at all. At that point, the model has learned the best-fit line it can find.

**In short:**

First we select an initial value for the parameters. Then, at each step, we update the parameter using the formula:


$$
param_{new} =  param_{old} -\alpha \frac{\partial J}{\partial param_{old}}
$$
$\alpha$ is the learning rate. It controls how large a step we take toward the minimum loss function at each update.

We repeat this process until the change between the old and new parameter becomes small enough (convergence), or until a fixed number of iterations (epochs) has been completed.

Now lets implement everything in the code

##### First we have to generate the data to to train the model

In [3]:
import numpy as np
np.random.seed(42)
X = 10*np.random.rand(200) # it represents study hour from [0,10) 
# random.rand(n) gives n float from the uniform dist [0,1)
y = 4+ 3* X + np.random.randn(200) # it is the score output, score =  4 + 3x + error
# random.randn(n) gives n random number from normal distribution 

##### Now we do a train-test split so that we can train the model on the training data and then test the model with testing data

In [4]:
indices = np.random.permutation(len(X))
# np.random.permutation(n) takes the first n number and randomly shuffles it from 0 to n-1 
split = int(0.8 * len(X))
# int() takes a number inside the paranthesis and makes it whole. in this case as len(X) is 200 split will give 160. 
train_idx, test_idx = indices[:split], indices[split:]
# now we split the indices into training index and testing index (80:20).

X_train,X_test = X[train_idx], X[test_idx]
y_train,y_test = y[train_idx], y[test_idx]

# here we split the X and y var into training and testing set based on the indices that we split 

##### Initializing the parameters

In [5]:
w = 0
b= 0
# these are the initial values of beta and beta0. here beta1=w and beta0=b. we first take 0 as initial value 
learning_rate = 0.01
epochs = 2000 # the number of repeatation through the dataset

n = len(X_train)

##### Now we train the data using gradient descent

In [6]:
for epoch in range(epochs):
    # first we make predictions using w and b (old)
    y_pred = w *X_train +b 

    # then we calculate the MSE
    error = y_pred-y_train 
    loss = np.mean(error**2)

    # calculate gradients 
    dw = (2/n)*np.sum(error*X_train)
    db = (2/n)*np.sum(error) 
    # this is just the derivatives of the loss function with respect to the parameters 

    # now we update the paraemeters (new)
    w = w-learning_rate*dw 
    b = b-learning_rate*db 

    # now lets write a code that prints out the progress so that we can see how far the code has gone 
    if(epoch +1 ) % 400 ==0:
        print(f"Epoch: {epoch+1}, MSE: {loss:.4f}")
    

Epoch: 400, MSE: 0.9729
Epoch: 800, MSE: 0.9280
Epoch: 1200, MSE: 0.9275
Epoch: 1600, MSE: 0.9274
Epoch: 2000, MSE: 0.9274


##### Now we will evaluate the model on testing data

In [7]:
# first we will predict the output for the test dataset.
y_test_pred = w*X_test +b 

# then we do various tests with it. Mainly finding some loss functions and doing goodness of fit test 
mae = np.mean(np.abs(y_test - y_test_pred)) # mean absolute error

mse = np.mean((y_test - y_test_pred) ** 2) # mean squared error

rmse = np.sqrt(mse) # root mean squared error


# goodness of fit test 

sse = np.sum((y_test-y_test_pred)**2)
sst = np.sum((y_test-np.mean(y_test))**2)
r2 = 1- (sse/sst)

print("\nLearned model:")
print(f"y = {w:.3f}x + {b:.3f}")

print(f"\nMAE:  {mae:.3f}")
print(f"MSE:  {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")


Learned model:
y = 2.989x + 4.116

MAE:  0.704
MSE:  0.960
RMSE: 0.980
R²:   0.985


##### Now lets try to predict a new observation with our model

In [8]:
x_new = 5
prediction = w*x_new +b 

print(f"Prediction for new x: {x_new} is {prediction}")

Prediction for new x: 5 is 19.062288390813585
